In [ ]:


from typing import Dict, Any, Optional
import os
from datetime import datetime
import pandas as pd
import numpy as np
from langchain_openai import ChatOpenAI
from langchain.agents import Tool, initialize_agent, AgentType


# ---------------------
# 基础配置
# ---------------------
# 自动查找当前目录下的 example.xlsx 文件
for candidate in ["example.xlsx", "./example.xlsx", "D:/working/example.xlsx"]:
    if os.path.exists(candidate):
        EXCEL_PATH = candidate
        break
else:
    raise FileNotFoundError("未找到 example.xlsx 文件，请确认路径。")

SHEET_NAME = "Income Statement"

In [ ]:

#llm设定

LLM_CONFIG = {
    "model": "Qwen/Qwen2.5-72B-Instruct",
    "openai_api_key": "sk-uysqxidvfkuzylezjufcottmkodwkrxyvxaachhosyeswxya",
    "openai_api_base": "https://api.siliconflow.cn/v1",
    "temperature": 0.0,
    "max_tokens": 1024,
}

In [ ]:


# ---------------------
# 工具函数
# ---------------------
def parse_to_datetime(val) -> Optional[pd.Timestamp]:
    """智能解析日期"""
    if pd.isna(val):
        return None
    if isinstance(val, (pd.Timestamp, datetime)):
        return pd.Timestamp(val)
    s = str(val).strip()
    if s.isdigit():
        if len(s) == 8:
            return pd.to_datetime(s, format="%Y%m%d", errors="coerce")
        elif len(s) == 6:
            return pd.to_datetime(s + "01", format="%Y%m%d", errors="coerce")
        elif len(s) == 4:
            return pd.to_datetime(s + "0101", format="%Y%m%d", errors="coerce")
    return pd.to_datetime(s, errors="coerce")


def load_excel_file(path: str) -> Dict[str, pd.DataFrame]:
    """加载 Excel 文件所有 sheet"""
    if not os.path.exists(path):
        raise FileNotFoundError(f"文件不存在: {path}")
    xls = pd.ExcelFile(path)
    sheets = {}
    for sheet in xls.sheet_names:
        df = pd.read_excel(path, sheet_name=sheet)
        df.columns = [str(c).strip() for c in df.columns]
        sheets[sheet] = df
    return sheets


def detect_time_column(df: pd.DataFrame) -> Optional[str]:
    """自动识别时间列"""
    for col in df.columns:
        if "统计截止日期" in col or "日期" in col:
            return col
    return df.columns[0] if len(df.columns) > 0 else None


def ensure_time_index(df: pd.DataFrame, time_col: str) -> pd.DataFrame:
    """将时间列设为索引"""
    df2 = df.copy()
    df2[time_col] = df2[time_col].apply(parse_to_datetime)
    df2 = df2.sort_values(by=time_col)
    df2 = df2.set_index(time_col)
    return df2


def compute_series_metrics(series: pd.Series, periods_for_cagr: int = 5) -> Dict[str, Any]:
    """计算时间序列指标"""
    s = series.dropna().astype(float)
    out = {}
    if s.empty:
        return {"error": "无有效数据"}

    out["起始值"] = float(s.iloc[0])
    out["最新值"] = float(s.iloc[-1])

    pct = s.pct_change().dropna()
    if not pct.empty:
        out["平均增长率"] = float(pct.mean())
        out["增长波动率"] = float(pct.std())
    else:
        out["平均增长率"] = None
        out["增长波动率"] = None

    if isinstance(s.index, pd.DatetimeIndex):
        yearly = s.resample("YE").last()  # ✅ 修正为 YE
        yoy = yearly.pct_change().dropna()
        out["最近同比"] = float(yoy.iloc[-1]) if len(yoy) >= 1 else None

    n = min(periods_for_cagr, len(s) - 1)
    if n >= 1:
        start = s.iloc[-(n + 1)]
        end = s.iloc[-1]
        if start > 0 and end > 0:
            out["CAGR区间长度"] = n
            out["CAGR"] = (end / start) ** (1 / n) - 1
        else:
            out["CAGR"] = None
    return out


def compute_financial_ratios(df: pd.DataFrame) -> Dict[str, pd.Series]:
    """计算主要财务比率"""
    ratios = {}

In [ ]:

    # 确保所有关键列为数值类型
    for col in ["营业总收入", "营业收入", "利润总额", "净利润"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "营业总收入" in df.columns and "净利润" in df.columns:
        ratios["净利率"] = (df["净利润"] / df["营业总收入"]).replace([np.inf, -np.inf], np.nan)

    if "营业总收入" in df.columns and "利润总额" in df.columns:
        ratios["利润率"] = (df["利润总额"] / df["营业总收入"]).replace([np.inf, -np.inf], np.nan)

    if "营业总收入" in df.columns and "营业收入" in df.columns:
        ratios["营业收入占比"] = (df["营业收入"] / df["营业总收入"]).replace([np.inf, -np.inf], np.nan)

    return ratios

In [ ]:


# ---------------------
# 主分析函数
# ---------------------
def analyze_sheet_tool(path: str, sheet_name: str) -> Dict[str, Any]:
    """主分析入口"""
    sheets = load_excel_file(path)
    if sheet_name not in sheets:
        return {"error": f"未找到 sheet: {sheet_name}"}
    df = sheets[sheet_name]
    time_col = detect_time_column(df)
    df_t = ensure_time_index(df, time_col)

    key_cols = ["营业总收入", "营业收入", "利润总额", "净利润"]
    metrics = {}
    for col in key_cols:
        if col in df_t.columns:
            ser = pd.to_numeric(df_t[col], errors="coerce")
            metrics[col] = compute_series_metrics(ser)

    ratios = compute_financial_ratios(df_t)
    ratio_summary = {}
    for name, s in ratios.items():
        s = s.dropna()
        ratio_summary[name] = {
            "最新值": float(s.iloc[-1]) if len(s) > 0 else None,
            "样本数": len(s)
        }

    return {
        "sheet": sheet_name,
        "行数": df.shape[0],
        "列数": df.shape[1],
        "时间列": time_col,
        "主要指标分析": metrics,
        "财务比率": ratio_summary
    }

In [ ]:


# 主程序入口
if __name__ == "__main__":
    print("=== 财务报表分析（中文列名版） ===")
    print(f"文件路径: {EXCEL_PATH}\n")

    result = analyze_sheet_tool(EXCEL_PATH, SHEET_NAME)
    import json
    print(json.dumps(result, indent=2, ensure_ascii=False))
